Import necessary packages:

In [1]:
import itertools
from sage.all import *

# Compute multiplication table for $B(n)$:

We use the matrix representation of $P$ from the Gardam paper (https://arxiv.org/abs/2312.05240), which makes solving the word problem in $P$ reduce to matrix multiplication over integer matrices.

In [2]:
K = GF(2)
gens = [matrix(QQ, 4, [1, 0, 0, 1, 0, -1, 0, 1, 0, 0, -1, 0, 0, 0, 0, 1]), matrix(ZZ, 4, [-1, 0, 0, 0, 0, 1, 0, 1, 0, 0, -1, 1, 0, 0, 0, 1])]
P = MatrixGroup(gens)

symbols = [P.one(), P.gen(0), P.gen(0).inverse(), P.gen(1), P.gen(1).inverse()]
B_5 = [prod(word) for word in itertools.product(symbols, repeat=5)]
B_5 = list(set(B_5))
print(len(B_5))
print(B_5[0])

147
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]


In [3]:
# Keys are the product, values are lists of pairs realizing the product
product_table = dict()
for i,j in itertools.product(range(len(B_5)), repeat=2):
    a, b = B_5[i], B_5[j]
    val = a*b
    if val in product_table:
        product_table[val].append((i, j))
    else:
        product_table[val] = [(i,j),]
print(len(product_table))

981


# Assert System of Boolean Equations:

In [4]:
from pysat.formula import *
from pysat.solvers import *

In [5]:
a_vars = [Atom(f"a_{i}") for i in range(len(B_5))]
b_vars = [Atom(f"b_{j}") for j in range(len(B_5))]

x_vars = dict()
cnf = CNF()

# Non triviality
formula = Equals(a_vars[0], PYSAT_TRUE)
cnf.extend([c for c in formula])

formula = Or(*list(a_vars[i] for i in range(1, len(B_5))))
cnf.extend([c for c in formula])

# Product equations x_g,h = a_g * b_h
for i,j in itertools.product(range(len(B_5)), repeat=2):
    x_vars[(i, j)] = Atom(f"x_{i}{j}")
    formula = And(Implies(x_vars[(i,j)], a_vars[i]), Implies(x_vars[(i,j)], b_vars[j]), Implies(And(a_vars[i], b_vars[j]), x_vars[(i,j)]))
    cnf.extend([c for c in formula])

# Sum equations sum_{gh=k}(x_g,h) = delta(1,k) for each k in the product table
if len(product_table[P.one()]) > 1:
    formula = XOr(*[x_vars[(i,j)] for i,j in product_table[P.one()]])
    cnf.extend([c for c in formula])
else:
    formula = Equals(x_vars[product_table[P.one()][0]], PYSAT_TRUE)
    cnf.extend([c for c in formula])

for val in product_table:
    if val.is_one():
        continue
    if len(product_table[val]) > 1:
        formula = Neg(XOr(*[x_vars[(i,j)] for i,j in product_table[val]]))
        cnf.extend([c for c in formula])
    else:
        formula = Equals(x_vars[product_table[val][0]], PYSAT_FALSE)
        cnf.extend([c for c in formula])


In [ ]:
solution = []
with Minisat22(bootstrap_with=cnf.clauses) as m:
    print(m.solve())
    solution = m.get_model()
support = list(filter(lambda n: n > 0, solution))
# print(support)
# print(solution)

In [ ]:
obj2id = Formula.export_vpool().obj2id

for i in range(len(B_5)):
    if obj2id[a_vars[i]] in support:
        print(f"a_{i}")
    if obj2id[b_vars[i]] in support:
        print(f"b_{i}")

for i,j in itertools.product(range(len(B_5)), repeat=2):
    if obj2id[x_vars[(i,j)]] in support:
        print(f"x_{i},{j}")